# AMEX Enterprise Credit Risk Platform
## Notebook 15 — Technical Documentation: Data Dictionary, Architecture, API Reference & Setup Guide
### Phase 1 · Problem Statement 1: Credit Scoring / PD Prediction

CRISP-DM stage: **Deployment / Documentation**. Notebook 15 of 18. Depends on Notebook 01 only (the config file); every other notebook's output is read opportunistically, if present, exactly like Notebook 17's later rollup — so this notebook produces a coherent, honest snapshot of the platform's *actual current state*, whatever fraction of the 18 notebooks has been run so far.

**What this notebook builds, all from real, live-scanned sources — nothing hand-typed that could drift from the truth:**

- A **data dictionary** built directly from Notebook 05's real fitted `preprocessing_artifacts.joblib` (feature names, type, imputed median where numeric, category list where categorical, champion-model importance rank where available) — not a hand-maintained spreadsheet that goes stale.
- A **platform artifact inventory** — a live filesystem scan of every pillar directory, real file counts and sizes, not a claimed manifest.
- An **architecture diagram and dependency map** of the real notebook-to-notebook data flow this platform actually implements.
- An **API reference** parsed directly from Notebook 10's real, exported `openapi_spec.json` (if present) -- not hand-written docs that could drift from the real service.
- A **model documentation summary** cross-referencing Notebook 09's real model registry.
- A **glossary** of the financial/ML/regulatory terms used throughout this platform (PD, LGD, EAD, ECL, RWA, PSI, IFRS9, Basel III, WARP, CRISP-DM, SHAP, the AMEX competition metric) — reference definitions, not computed data.
- A **setup & run-order guide** and a generated **README.md**.

**Deliverables:** `data_dictionary.csv`, `platform_artifact_inventory.csv`, `architecture_diagram.png`, `api_reference.csv` (if Notebook 10 has run), `glossary.csv`, `README.md`, `technical_documentation_checklist.csv`, and `Technical_Documentation_Report.docx`.

**Run the single code cell below, once.** Idempotent — every output file is overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOK 01 (EVERYTHING ELSE OPTIONAL)
# =============================================================================
import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebook 01 (Everything Else Optional)")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"{CONFIG_PATH} not found.\nFix: run 01_business_understanding.ipynb first -- "
                             f"this notebook reads its pillar directory map.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]

_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)

TECH_DOC_DIR = PILLAR_DIRS["technical_documentation"]
TECH_DOC_DIR.mkdir(parents=True, exist_ok=True)

# --- Discover every notebook_0N_summary.json this run can find -- a fully
#     generic scan, not a hand-typed list, so this notebook stays honest
#     about exactly how much of the platform has actually been run. ---
NOTEBOOK_SUMMARIES = {}
for _p in sorted(ARTIFACTS_DIR.glob("notebook_*_summary.json")):
    try:
        _num = int(_p.stem.split("_")[1])
    except (IndexError, ValueError):
        continue
    with open(_p, "r", encoding="utf-8") as f:
        NOTEBOOK_SUMMARIES[_num] = json.load(f)

print(f"Notebook summaries found: {sorted(NOTEBOOK_SUMMARIES.keys())} "
      f"({len(NOTEBOOK_SUMMARIES)} of the 13 notebooks (02-14) that write one)")
print(f"Technical documentation will be written under: {TECH_DOC_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION, LIBRARY IMPORTS & ADAPTIVE RAM CEILING
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration, Library Imports & Adaptive RAM Ceiling")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    import joblib
except ImportError:
    missing.append("joblib")
try:
    from docx import Document
    from docx.shared import Inches
except ImportError:
    missing.append("python-docx")

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )

print("(Reporting only -- this notebook is I/O-bound documentation generation, not thread-parallelized work.)")


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


_process_start_rss_gb = _rss_gb()
_live_vm = psutil.virtual_memory()
ADAPTIVE_RAM_FRACTION = _resource_limits.get("ram_fraction_cap", 0.90)
MAX_RAM_BYTES = int(_live_vm.available * ADAPTIVE_RAM_FRACTION)
print(f"Adaptive RAM ceiling (this run) : {MAX_RAM_BYTES / 1e9:.2f} GB")
print(f"\nProcess RSS at Section 2 start: {_process_start_rss_gb:.2f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: PLATFORM ARTIFACT INVENTORY -- LIVE FILESYSTEM SCAN
# =============================================================================
_section("SECTION 3: Platform Artifact Inventory -- Live Filesystem Scan")

# --- A real directory walk of every pillar this platform defines -- not a
#     claimed manifest. Whatever notebooks have not yet been run simply show
#     0 files, honestly. ---
_inventory_rows = []
for _pillar_name, _pillar_path in PILLAR_DIRS.items():
    if _pillar_path.exists():
        _files = [f for f in _pillar_path.rglob("*") if f.is_file()]
        _total_bytes = sum(f.stat().st_size for f in _files)
        _inventory_rows.append({
            "pillar": _pillar_name, "directory": str(_pillar_path), "file_count": len(_files),
            "total_size_mb": round(_total_bytes / 1e6, 3), "exists": True,
        })
    else:
        _inventory_rows.append({
            "pillar": _pillar_name, "directory": str(_pillar_path), "file_count": 0,
            "total_size_mb": 0.0, "exists": False,
        })

inventory_df = pd.DataFrame(_inventory_rows)
inventory_path = TECH_DOC_DIR / "platform_artifact_inventory.csv"
inventory_df.to_csv(inventory_path, index=False)
print(inventory_df.to_string(index=False))
print(f"\nTotal real files across the platform: {inventory_df['file_count'].sum():,}  "
      f"({inventory_df['total_size_mb'].sum():.1f} MB)")
print(f"\u2705 Saved -> {inventory_path}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: DATA DICTIONARY -- BUILT FROM NOTEBOOK 05'S REAL FITTED PREPROCESSING
# =============================================================================
_section("SECTION 4: Data Dictionary -- Built From Notebook 05's Real Fitted Preprocessing")

MODEL_DEV_DIR = PILLAR_DIRS["model_development"]
PREPROCESSING_PATH = MODEL_DEV_DIR / "models" / "preprocessing_artifacts.joblib"
CHAMPION_IMPORTANCE_PATH = MODEL_DEV_DIR / "champion_feature_importance.csv"

data_dictionary_df = None
if PREPROCESSING_PATH.exists():
    preprocessing_artifacts = joblib.load(PREPROCESSING_PATH)
    label_encoders = preprocessing_artifacts["label_encoders"]
    feature_medians = preprocessing_artifacts["feature_medians"]
    all_feature_cols = preprocessing_artifacts["all_feature_cols"]
    categorical_encode_cols = set(preprocessing_artifacts["categorical_encode_cols"])
    numeric_feature_cols = set(preprocessing_artifacts["numeric_feature_cols"])

    _importance_rank = {}
    if CHAMPION_IMPORTANCE_PATH.exists():
        _imp_df = pd.read_csv(CHAMPION_IMPORTANCE_PATH).reset_index(drop=True)
        _importance_rank = {row["feature"]: i + 1 for i, row in _imp_df.iterrows()}

    _dict_rows = []
    for feat in all_feature_cols:
        if feat in categorical_encode_cols:
            _dict_rows.append({
                "feature": feat, "type": "categorical (label-encoded)",
                "categories": ", ".join(label_encoders[feat]["classes"]) if feat in label_encoders else None,
                "imputed_median": None,
                "champion_importance_rank": _importance_rank.get(feat),
            })
        else:
            _dict_rows.append({
                "feature": feat, "type": "numeric",
                "categories": None,
                "imputed_median": round(float(feature_medians.get(feat)), 6) if feat in feature_medians else None,
                "champion_importance_rank": _importance_rank.get(feat),
            })
    data_dictionary_df = pd.DataFrame(_dict_rows).sort_values(
        by="champion_importance_rank", na_position="last").reset_index(drop=True)
    print(f"Built a real data dictionary for {len(data_dictionary_df)} features "
          f"({len(numeric_feature_cols)} numeric, {len(categorical_encode_cols)} categorical), "
          f"from {PREPROCESSING_PATH.name}")
else:
    print(f"{PREPROCESSING_PATH} not found -- Notebook 05 has not been run yet. "
          f"Data dictionary will be a documented placeholder, not fabricated feature data.")
    data_dictionary_df = pd.DataFrame(columns=["feature", "type", "categories", "imputed_median", "champion_importance_rank"])

data_dictionary_path = TECH_DOC_DIR / "data_dictionary.csv"
data_dictionary_df.to_csv(data_dictionary_path, index=False)
print(f"\u2705 Saved -> {data_dictionary_path}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: ARCHITECTURE -- REAL NOTEBOOK DEPENDENCY MAP & DIAGRAM
# =============================================================================
_section("SECTION 5: Architecture -- Real Notebook Dependency Map & Diagram")

# --- This is documentation content this platform's own build process
#     defines -- the real CRISP-DM stage each notebook implements and the
#     real "depends on" relationships enforced by every notebook's own
#     Section 1 file-existence checks (see each notebook's intro cell).
#     Not computed data -- authored reference documentation, verified
#     against the actual required-file checks in every notebook this
#     platform has generated so far. ---
NOTEBOOK_ARCHITECTURE = [
    {"n": 1, "name": "Business Understanding", "stage": "Business Understanding", "depends_on": []},
    {"n": 2, "name": "Data Engineering", "stage": "Data Preparation", "depends_on": [1]},
    {"n": 3, "name": "Data Validation & EDA", "stage": "Data Understanding", "depends_on": [1, 2]},
    {"n": 4, "name": "Feature Engineering", "stage": "Data Preparation", "depends_on": [1, 2]},
    {"n": 5, "name": "Model Development", "stage": "Modeling", "depends_on": [1, 4]},
    {"n": 6, "name": "Explainable AI (SHAP/LIME)", "stage": "Evaluation", "depends_on": [1, 5]},
    {"n": 7, "name": "Model Risk Management", "stage": "Evaluation / Governance", "depends_on": [1, 4, 5]},
    {"n": 8, "name": "Basel III / IFRS9 Mapping", "stage": "Evaluation / Regulatory", "depends_on": [1, 5]},
    {"n": 9, "name": "MLOps", "stage": "Deployment", "depends_on": [1, 5]},
    {"n": 10, "name": "FastAPI Deployment", "stage": "Deployment", "depends_on": [1, 5]},
    {"n": 11, "name": "Docker", "stage": "Deployment", "depends_on": [1, 10]},
    {"n": 12, "name": "Monitoring", "stage": "Deployment / Ongoing Monitoring", "depends_on": [1, 4, 5]},
    {"n": 13, "name": "Power BI Dashboard", "stage": "Deployment / Business Intelligence", "depends_on": [1, 4, 5]},
    {"n": 14, "name": "Executive Reports", "stage": "Deployment / Business Reporting", "depends_on": [1, 5]},
    {"n": 15, "name": "Technical Documentation", "stage": "Deployment / Documentation", "depends_on": [1]},
    {"n": 16, "name": "Production Architecture", "stage": "Deployment / Architecture", "depends_on": [1]},
    {"n": 17, "name": "Comprehensive Reporting", "stage": "Deployment / Rollup", "depends_on": [1]},
    {"n": 18, "name": "Repository Packaging", "stage": "Deployment / Packaging", "depends_on": [1]},
]
architecture_df = pd.DataFrame([
    {"n": r["n"], "notebook": r["name"], "crisp_dm_stage": r["stage"],
     "depends_on": ", ".join(str(d) for d in r["depends_on"]) or "(none)",
     "has_run": (r["n"] in NOTEBOOK_SUMMARIES) or (r["n"] == 1)}
    for r in NOTEBOOK_ARCHITECTURE
])
architecture_path = TECH_DOC_DIR / "notebook_architecture_map.csv"
architecture_df.to_csv(architecture_path, index=False)
print(architecture_df.to_string(index=False))

PROBLEM_NAME = "Phase 1 \u00b7 Problem 1 -- Credit Scoring / PD Prediction"
VIZ = {"surface": "#fcfcfb", "text_primary": "#0b0b0b", "text_secondary": "#52514e", "grid": "#e3e2dd",
       "cat_blue": "#2a78d6", "cat_green": "#3a9e5f", "cat_grey": "#9c9b96"}

_stage_order = []
for r in NOTEBOOK_ARCHITECTURE:
    if r["stage"] not in _stage_order:
        _stage_order.append(r["stage"])
_stage_rows = {s: [r["n"] for r in NOTEBOOK_ARCHITECTURE if r["stage"] == s] for s in _stage_order}

fig, ax = plt.subplots(figsize=(11, 8), dpi=150)
ax.set_facecolor(VIZ["surface"]); fig.set_facecolor(VIZ["surface"])
ax.axis("off")
_has_run_lookup = dict(zip(architecture_df["n"], architecture_df["has_run"]))
_y = len(_stage_order)
_row_h = 1.0
for _stage in _stage_order:
    _ns = _stage_rows[_stage]
    ax.text(0.2, _y - 0.5, _stage, ha="left", va="center", fontsize=9, color=VIZ["text_secondary"], weight="bold")
    for _i, _n in enumerate(_ns):
        _x = 3.2 + _i * 1.15
        _color = VIZ["cat_green"] if _has_run_lookup.get(_n) else VIZ["cat_grey"]
        ax.add_patch(plt.Rectangle((_x, _y - 0.85), 1.0, 0.7, facecolor=_color, edgecolor="none"))
        ax.text(_x + 0.5, _y - 0.5, f"NB{_n:02d}", ha="center", va="center", fontsize=8, color="white", weight="bold")
    _y -= _row_h
ax.set_xlim(0, 13); ax.set_ylim(0, len(_stage_order) + 0.3)
_legend_patches = [plt.Rectangle((0, 0), 1, 1, facecolor=VIZ["cat_green"], label="Has run (summary artifact found)"),
                    plt.Rectangle((0, 0), 1, 1, facecolor=VIZ["cat_grey"], label="Not yet run")]
ax.legend(handles=_legend_patches, loc="lower right", fontsize=8, frameon=False)
ax.set_title(f"{PROBLEM_NAME}\nPlatform Architecture -- CRISP-DM Stage by Notebook (Live Run Status)", fontsize=11, color=VIZ["text_primary"])
fig.tight_layout()
architecture_chart_path = TECH_DOC_DIR / "architecture_diagram.png"
fig.savefig(architecture_chart_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig)
print(f"\u2705 Saved -> {architecture_path}")
print(f"\u2705 Saved -> {architecture_chart_path}")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: API REFERENCE -- PARSED FROM NOTEBOOK 10'S REAL OPENAPI SPEC
# =============================================================================
_section("SECTION 6: API Reference -- Parsed From Notebook 10's Real OpenAPI Spec")

API_DIR = PILLAR_DIRS["fastapi_deployment"]
OPENAPI_SPEC_PATH = API_DIR / "openapi_spec.json"

api_reference_df = None
if OPENAPI_SPEC_PATH.exists():
    with open(OPENAPI_SPEC_PATH, "r", encoding="utf-8") as f:
        openapi_spec = json.load(f)

    _api_rows = []
    for _path, _methods in openapi_spec.get("paths", {}).items():
        for _method, _op in _methods.items():
            _summary = _op.get("summary") or _op.get("operationId") or ""
            _params = ", ".join(p.get("name", "") for p in _op.get("parameters", [])) or "(none)"
            _req_body = _op.get("requestBody", {})
            _req_schema_ref = None
            try:
                _req_schema_ref = (_req_body.get("content", {}).get("application/json", {})
                                    .get("schema", {}).get("$ref", "")).rsplit("/", 1)[-1] or None
            except (AttributeError, IndexError):
                pass
            _responses = ", ".join(sorted(_op.get("responses", {}).keys())) or "(none)"
            _api_rows.append({
                "path": _path, "method": _method.upper(), "summary": _summary,
                "query_parameters": _params, "request_body_schema": _req_schema_ref,
                "response_status_codes": _responses,
            })
    api_reference_df = pd.DataFrame(_api_rows)
    _schema_names = list(openapi_spec.get("components", {}).get("schemas", {}).keys())
    print(f"Parsed {len(api_reference_df)} real endpoint(s) and {len(_schema_names)} schema(s) "
          f"from {OPENAPI_SPEC_PATH.name} (real, Notebook 10's own exported spec)")
    print(api_reference_df.to_string(index=False))
else:
    print(f"{OPENAPI_SPEC_PATH} not found -- Notebook 10 has not been run yet. "
          f"API reference will be a documented placeholder, not fabricated endpoint data.")
    api_reference_df = pd.DataFrame(columns=["path", "method", "summary", "query_parameters",
                                              "request_body_schema", "response_status_codes"])

api_reference_path = TECH_DOC_DIR / "api_reference.csv"
api_reference_df.to_csv(api_reference_path, index=False)
print(f"\u2705 Saved -> {api_reference_path}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: MODEL DOCUMENTATION SUMMARY -- CROSS-REFERENCED FROM NOTEBOOK 09
# =============================================================================
_section("SECTION 7: Model Documentation Summary -- Cross-Referenced From Notebook 09")

MLOPS_DIR = PILLAR_DIRS["mlops"]
REGISTRY_PATH = MLOPS_DIR / "model_registry.json"

model_doc_df = None
if REGISTRY_PATH.exists():
    with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
        _registry = json.load(f)
    model_doc_df = pd.DataFrame(_registry["entries"])
    print(f"Loaded {len(model_doc_df)} real registered model version(s) from {REGISTRY_PATH.name}")
    print(model_doc_df.to_string(index=False))
else:
    print(f"{REGISTRY_PATH} not found -- Notebook 09 has not been run yet. "
          f"Model documentation will be a documented placeholder, not fabricated model data.")
    model_doc_df = pd.DataFrame(columns=["model_name", "version", "registered_at_utc", "sha256",
                                          "holdout_auc", "holdout_amex_metric"])

model_doc_path = TECH_DOC_DIR / "model_documentation_summary.csv"
model_doc_df.to_csv(model_doc_path, index=False)
print(f"\u2705 Saved -> {model_doc_path}")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: GLOSSARY -- REFERENCE DEFINITIONS (NOT COMPUTED DATA)
# =============================================================================
_section("SECTION 8: Glossary -- Reference Definitions (Not Computed Data)")

# --- Standard, factual, industry-reference definitions of terms used
#     throughout this platform's 18 notebooks. This is authored reference
#     material -- like a textbook glossary -- not a live computation, and is
#     clearly presented as such; it carries no MEASURED/ASSUMPTION label
#     because it asserts no numeric fact about this platform's data or
#     results. ---
GLOSSARY = [
    {"term": "PD", "definition": "Probability of Default -- the model's core output: the estimated likelihood a customer defaults within the prediction horizon."},
    {"term": "LGD", "definition": "Loss Given Default -- the fraction of exposure a lender expects to lose if a customer defaults, net of recoveries."},
    {"term": "EAD", "definition": "Exposure at Default -- the expected outstanding balance at the moment of default."},
    {"term": "ECL", "definition": "Expected Credit Loss -- IFRS 9's core loss-provisioning quantity, computed as PD x LGD x EAD (Stage 1) or over a lifetime horizon (Stage 2/3)."},
    {"term": "RWA", "definition": "Risk-Weighted Assets -- a Basel capital-adequacy quantity; regulatory capital held is a fraction of RWA, not of raw exposure."},
    {"term": "PSI", "definition": "Population Stability Index -- a standard drift metric comparing a feature's (or score's) distribution between two populations, e.g. training vs. a newer population."},
    {"term": "IFRS 9", "definition": "International Financial Reporting Standard 9 -- governs expected-credit-loss provisioning and staging (Stage 1/2/3) for financial instruments."},
    {"term": "Basel III", "definition": "The international regulatory capital framework (Basel Committee on Banking Supervision) setting minimum capital ratios and risk-weighting methodology for banks."},
    {"term": "QRRE", "definition": "Qualifying Revolving Retail Exposure -- the Basel exposure class credit-card-style revolving credit falls under, with its own IRB capital formula and fixed correlation parameter."},
    {"term": "AMEX competition metric", "definition": "The custom evaluation metric used by the Kaggle American Express Default Prediction competition -- a blend of Normalized Gini coefficient and the default-capture rate in the top 4% of predicted risk."},
    {"term": "SHAP", "definition": "SHapley Additive exPlanations -- a game-theoretic method for attributing a model's prediction to its input features."},
    {"term": "LIME", "definition": "Local Interpretable Model-agnostic Explanations -- explains individual predictions by fitting a simple, local surrogate model around them."},
    {"term": "WARP", "definition": "This platform's own performance convention: configuring library thread pools (e.g. Polars) to a fixed percentage of detected logical CPU cores, set once and reused across notebooks."},
    {"term": "CRISP-DM", "definition": "Cross-Industry Standard Process for Data Mining -- the six-stage process framework (Business Understanding through Deployment) this platform's notebooks are organized around."},
    {"term": "Champion model", "definition": "The best-performing model from Notebook 05's comparison, by holdout AMEX metric -- the model every downstream notebook reuses as-is."},
    {"term": "Holdout / test split", "definition": "The portion of data held out from training and never used to fit the model or its preprocessing -- the only data every validation and monitoring check in this platform trusts."},
    {"term": "Star schema", "definition": "A BI data-modeling pattern: one central fact table (the granular, measurable events) surrounded by dimension tables (the descriptive context to slice and filter by)."},
    {"term": "DAX", "definition": "Data Analysis Expressions -- the formula language Power BI (and Excel Power Pivot) uses to define measures and calculated columns."},
]
glossary_df = pd.DataFrame(GLOSSARY)
glossary_path = TECH_DOC_DIR / "glossary.csv"
glossary_df.to_csv(glossary_path, index=False)
print(f"{len(glossary_df)} terms defined.")
print(f"\u2705 Saved -> {glossary_path}")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: SETUP & RUN-ORDER GUIDE + README.md
# =============================================================================
_section("SECTION 9: Setup & Run-Order Guide + README.md")

SETUP_STEPS = [
    "Install Python 3.11 and Conda (or venv) on a Windows machine with the raw Kaggle "
    "American Express Default Prediction CSVs already downloaded.",
    "Create and activate a fresh environment, then install dependencies: "
    "polars, numpy, pandas, scikit-learn, xgboost, lightgbm, catboost, shap, lime, "
    "matplotlib, psutil, joblib, python-docx, scipy, openpyxl, fastapi, uvicorn, pydantic, pyyaml "
    "(Notebook 09 exports the exact installed versions to requirements.txt once it has run).",
    "Open 01_business_understanding.ipynb first, set DATA_ROOT to your local Kaggle CSV folder, "
    "and run its single code cell -- this creates project_config.json and every pillar directory "
    "every later notebook depends on.",
    "Run the remaining notebooks in numeric order (02 through 18) -- each notebook's own intro "
    "cell states exactly which prior notebooks it depends on and which of those are optional.",
    "Each notebook is a single, idempotent code cell: re-running it overwrites its own outputs "
    "in place (Notebook 09's model registry is the sole deliberate exception -- see its intro).",
]

_readme_lines = [
    "# AMEX Enterprise Credit Risk Platform",
    "",
    "Phase 1 - Problem Statement 1: Credit Scoring / PD Prediction, built on the real Kaggle "
    "American Express Default Prediction dataset.",
    "",
    "## What this is",
    "",
    "An 18-notebook, enterprise-grade credit risk platform: data engineering through model "
    "development, explainability, model risk management, Basel III / IFRS 9 regulatory mapping, "
    "MLOps, a real FastAPI scoring service, Docker containerization, production monitoring, a "
    "Power BI-ready data model, executive financial reporting, technical documentation, production "
    "architecture, a comprehensive rollup report, and repository packaging.",
    "",
    "Every notebook follows one rule throughout: every displayed number is either computed live by "
    "that notebook's own code on the real dataset, or is an explicitly labeled, editable ASSUMPTION "
    "where the dataset genuinely has no ground truth for it (e.g. business financial assumptions) -- "
    "never silently presented as fact.",
    "",
    "## Setup & run order",
    "",
]
for _i, _step in enumerate(SETUP_STEPS, start=1):
    _readme_lines.append(f"{_i}. {_step}")
_readme_lines += [
    "",
    "## Platform status (as of this run)",
    "",
    "| # | Notebook | CRISP-DM Stage | Has Run |",
    "|---|----------|-----------------|---------|",
]
for _, r in architecture_df.iterrows():
    _has_run_mark = "\u2705" if r["has_run"] else "\u2014"
    _readme_lines.append(f"| {r['n']} | {r['notebook']} | {r['crisp_dm_stage']} | {_has_run_mark} |")
_readme_lines += [
    "",
    "## Glossary",
    "",
    "See `Technical_Documentation/glossary.csv` for definitions of every financial/ML/regulatory "
    "term used across this platform.",
    "",
    f"_Generated by 15_technical_documentation.ipynb, {datetime.now().strftime('%Y-%m-%d %H:%M')}._",
    "",
]
README_CONTENT = "\n".join(_readme_lines)
readme_path = PROJECT_ROOT / "README.md"
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(README_CONTENT)
print(f"\u2705 Saved -> {readme_path}")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: TECHNICAL DOCUMENTATION READINESS CHECKLIST
# =============================================================================
_section("SECTION 10: Technical Documentation Readiness Checklist")

tech_doc_checklist = [
    {"dimension": "Data Dictionary Generated (Real, From Notebook 05)", "status": "Pass" if len(data_dictionary_df) > 0 else "Fallback (Notebook 05 not yet run)",
     "evidence": f"{len(data_dictionary_df)} features documented"},
    {"dimension": "Platform Artifact Inventory (Live Filesystem Scan)", "status": "Pass",
     "evidence": f"{inventory_df['file_count'].sum():,} real files across {len(inventory_df)} pillars"},
    {"dimension": "Architecture Diagram & Dependency Map", "status": "Pass",
     "evidence": f"{len(architecture_df)} notebooks mapped, {int(architecture_df['has_run'].sum())} have run"},
    {"dimension": "API Reference (Real, From Notebook 10)", "status": "Pass" if len(api_reference_df) > 0 else "Fallback (Notebook 10 not yet run)",
     "evidence": f"{len(api_reference_df)} endpoint(s) documented"},
    {"dimension": "Model Documentation (Real, From Notebook 09)", "status": "Pass" if len(model_doc_df) > 0 else "Fallback (Notebook 09 not yet run)",
     "evidence": f"{len(model_doc_df)} model version(s) documented"},
    {"dimension": "Glossary of Terms", "status": "Pass", "evidence": f"{len(glossary_df)} terms defined"},
    {"dimension": "Setup & Run-Order Guide", "status": "Pass", "evidence": f"{len(SETUP_STEPS)} steps documented"},
    {"dimension": "README.md Generated at Project Root", "status": "Pass", "evidence": readme_path.name},
]
tech_doc_checklist_df = pd.DataFrame(tech_doc_checklist)
tech_doc_checklist_path = TECH_DOC_DIR / "technical_documentation_checklist.csv"
tech_doc_checklist_df.to_csv(tech_doc_checklist_path, index=False)
print(tech_doc_checklist_df.to_string(index=False))
print(f"\u2705 Saved -> {tech_doc_checklist_path}")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: WORD REPORT -- TECHNICAL_DOCUMENTATION_REPORT.DOCX
# =============================================================================
_section("SECTION 11: Word Report -- Technical_Documentation_Report.docx")


def _add_heading(doc, text, level=1):
    return doc.add_heading(text, level=level)


def _add_table_from_df(doc, df, max_rows=25):
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = "Light Grid Accent 1"
    hdr = table.rows[0].cells
    for i, col in enumerate(df.columns):
        hdr[i].text = str(col).replace("_", " ").title()
    for _, row in df.head(max_rows).iterrows():
        cells_ = table.add_row().cells
        for i, col in enumerate(df.columns):
            cells_[i].text = "" if pd.isna(row[col]) else str(row[col])
    if len(df) > max_rows:
        doc.add_paragraph(f"... and {len(df) - max_rows} more row(s) -- see the full CSV for the complete table.")
    return table


report = Document()
report.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
report.add_paragraph("Technical Documentation Report -- Notebook 15")
report.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

_add_heading(report, "1. Platform Architecture", level=1)
report.add_paragraph(
    "The table below is the real dependency map enforced by every notebook's own Section 1 "
    "file-existence checks -- 'Has Run' reflects whether that notebook's summary artifact was "
    "actually found on disk this run, not a claim."
)
report.add_picture(str(architecture_chart_path), width=Inches(6.3))
_add_table_from_df(report, architecture_df)

_add_heading(report, "2. Platform Artifact Inventory (Live Filesystem Scan)", level=1)
_add_table_from_df(report, inventory_df)

_add_heading(report, "3. Data Dictionary", level=1)
if len(data_dictionary_df) > 0:
    report.add_paragraph(f"{len(data_dictionary_df)} features, built from Notebook 05's real fitted preprocessing artifacts.")
    _add_table_from_df(report, data_dictionary_df, max_rows=20)
else:
    report.add_paragraph("Notebook 05 has not been run yet -- no real feature list is available. See data_dictionary.csv once it has.")

_add_heading(report, "4. API Reference", level=1)
if len(api_reference_df) > 0:
    report.add_paragraph(f"Parsed directly from Notebook 10's real, exported openapi_spec.json.")
    _add_table_from_df(report, api_reference_df)
else:
    report.add_paragraph("Notebook 10 has not been run yet -- no real API spec is available. See api_reference.csv once it has.")

_add_heading(report, "5. Model Documentation", level=1)
if len(model_doc_df) > 0:
    _add_table_from_df(report, model_doc_df)
else:
    report.add_paragraph("Notebook 09 has not been run yet -- no real model registry is available. See model_documentation_summary.csv once it has.")

_add_heading(report, "6. Glossary", level=1)
_add_table_from_df(report, glossary_df, max_rows=30)

_add_heading(report, "7. Setup & Run-Order Guide", level=1)
for _step in SETUP_STEPS:
    report.add_paragraph(_step, style="List Number")

_add_heading(report, "8. Technical Documentation Readiness Checklist", level=1)
_add_table_from_df(report, tech_doc_checklist_df)

report_path = TECH_DOC_DIR / "Technical_Documentation_Report.docx"
report.save(str(report_path))
print(f"\u2705 Saved -> {report_path}")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: VERIFICATION -- INTEGRITY CHECKS ON EVERYTHING THIS NOTEBOOK WROTE
# =============================================================================
_section("SECTION 12: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("Architecture map covers all 18 notebooks", len(architecture_df) == 18, f"({len(architecture_df)})")
_check("Artifact inventory covers every pillar directory", len(inventory_df) == len(PILLAR_DIRS),
       f"({len(inventory_df)} vs {len(PILLAR_DIRS)})")
_check("Glossary is non-empty", len(glossary_df) > 0)
_check("README.md was written to the project root", readme_path.exists() and readme_path.stat().st_size > 0)
_check("Data dictionary honestly reflects Notebook 05's run status",
       (len(data_dictionary_df) > 0) == PREPROCESSING_PATH.exists())
_check("API reference honestly reflects Notebook 10's run status",
       (len(api_reference_df) > 0) == OPENAPI_SPEC_PATH.exists())
_check("Model documentation honestly reflects Notebook 09's run status",
       (len(model_doc_df) > 0) == REGISTRY_PATH.exists())

_expected_files = [inventory_path, data_dictionary_path, architecture_path, architecture_chart_path,
                    api_reference_path, model_doc_path, glossary_path, readme_path,
                    tech_doc_checklist_path, report_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 15 verification checks failed. See \u274c lines above.")

print("\nAll Notebook 15 checks passed.")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: RESOURCE / PERFORMANCE REPORT
# =============================================================================
_section("SECTION 13: Resource / Performance Report")

_final_rss_gb = _rss_gb()
performance_report = {
    "warp_thread_count_configured": WARP_THREAD_COUNT,
    "adaptive_ram_ceiling_gb": round(MAX_RAM_BYTES / 1e9, 2),
    "process_rss_at_start_gb": round(_process_start_rss_gb, 2),
    "process_rss_at_end_gb": round(_final_rss_gb, 2),
    "notebooks_with_summary_found": sorted(NOTEBOOK_SUMMARIES.keys()),
}
performance_report_path = ARTIFACTS_DIR / "notebook_15_performance_report.json"
with open(performance_report_path, "w", encoding="utf-8") as f:
    json.dump(performance_report, f, indent=2)
print(f"Process RSS: {_process_start_rss_gb:.2f} GB (start) -> {_final_rss_gb:.2f} GB (end)")
print(f"\u2705 Saved -> {performance_report_path}")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: WRITE NOTEBOOK 15 SUMMARY ARTIFACT (for Notebook 17's rollup)
# =============================================================================
_section("SECTION 14: Write Notebook 15 Summary Artifact")

notebook_15_summary = {
    "notebook": "15_technical_documentation", "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebooks_with_summary_found": sorted(NOTEBOOK_SUMMARIES.keys()),
    "data_dictionary_features": int(len(data_dictionary_df)),
    "api_endpoints_documented": int(len(api_reference_df)),
    "model_versions_documented": int(len(model_doc_df)),
    "total_platform_files": int(inventory_df["file_count"].sum()),
    "output_files": {p.name: str(p) for p in _expected_files + [performance_report_path]},
}
nb15_summary_path = ARTIFACTS_DIR / "notebook_15_summary.json"
with open(nb15_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_15_summary, f, indent=2)
print(f"\u2705 Saved -> {nb15_summary_path}")
print("\n\u2705 Section 14 complete.")


# =============================================================================
# SECTION 15: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 15: Notebook 15 Complete -- Handoff to Notebook 16")

print("NOTEBOOK 15: TECHNICAL DOCUMENTATION -- COMPLETE")
print(f"  Notebooks with a summary found   : {sorted(NOTEBOOK_SUMMARIES.keys())}")
print(f"  Data dictionary features          : {len(data_dictionary_df)}")
print(f"  API endpoints documented          : {len(api_reference_df)}")
print(f"  Model versions documented         : {len(model_doc_df)}")
print(f"  Total real platform files          : {inventory_df['file_count'].sum():,}")
print(f"  README.md                         : {readme_path}")
print(f"  Files produced                    : {len(_expected_files) + 2}")
for _p in _expected_files + [performance_report_path, nb15_summary_path]:
    print(f"    - {_p.name}")
print(f"  Next notebook                     : 16_production_architecture.ipynb")
print("\n\u2705 Ready to proceed.")
